In [63]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('word2vec_sample')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Miguel/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Miguel/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Miguel/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package word2vec_sample to
[nltk_data]     C:\Users\Miguel/nltk_data...
[nltk_data]   Package word2vec_sample is already up-to-date!


True

In [64]:
import nltk
import numpy as np
from nltk.corpus import wordnet as wn
from nltk.corpus import stopwords as sw
sw_english = sw.words('english')

In [65]:
def remove_sw(sentence):
    sentence = set(sentence)
    sentence.difference_update(set(sw_english))
    return sentence

In [66]:
def get_most_frequent_sense(word):
    synsets = wn.synsets(word) 
    if len(synsets) > 0:
        return synsets[0]
    return None

In [67]:
def compute_overlap(sentence, sense):
    definition = sense.definition().split()
    definition = remove_sw(definition)
    max_overlap = set(sentence).intersection(set(definition))
    if len(sense.examples()):
        for example in sense.examples():
            ex = example.split()
            ex = remove_sw(ex)
            overlap = set(sentence).intersection(set(ex))
            max_overlap = max_overlap.union(overlap)
    return max_overlap

In [68]:
def get_all_senses(word):
    return wn.synsets(word) 

In [69]:
def simplified_lesk_algorithm(word, sentence):
    sentence = sentence.split()
    sentence = remove_sw(sentence)
    best_sense = get_most_frequent_sense(word)
    max_overlap = compute_overlap(sentence, best_sense)
    for sense in get_all_senses(word):
        overlap = compute_overlap(sentence, sense)
        if overlap > max_overlap:
            max_overlap = overlap
            best_sense = sense
    return best_sense

In [70]:
print(simplified_lesk_algorithm('bank', 'Yesterday I went to the bank to withdraw the money and the credit card did not work').definition())

a financial institution that accepts deposits and channels the money into lending activities


Here we can see that, for the given sentence and the word to determine its sense, the best sense is related to financial language

In [71]:
import gensim
from nltk.data import find

# Cargar el modelo de embeding pre-entrenados del NLTK
word2vec_sample = str(find('models/word2vec_sample/pruned.word2vec.txt'))
model = gensim.models.KeyedVectors.load_word2vec_format(word2vec_sample, binary=False)


In [86]:
def get_sentence_word_embedding(sentence):
    embedding = np.zeros(300)
    for word in sentence:
        if model.has_index_for(word):
            embedding = np.add(embedding, model[word])
            #embedding = embedding/2
    return embedding

In [73]:
def cosinus_distance(e1, e2):
    return np.dot(e1, e2)/(np.sqrt(e1.dot(e1))*np.sqrt(e2.dot(e2)))

In [101]:
def WSD_with_embeddings(word, sentence):
    sentence = sentence.split()
    sentence = remove_sw(sentence)
    sentence_we = get_sentence_word_embedding(sentence)
    best_sense = ''
    max_overlap = 0
    for sense in get_all_senses(word):
        sense_def = remove_sw(sense.definition().split(' '))
        sense_def_we = get_sentence_word_embedding(sense_def)
        overlap = cosinus_distance(sentence_we, sense_def_we)
        if overlap > max_overlap:
            print('new best sense: ',sense.definition())
            print('overlap: ', overlap)
            print()
            max_overlap = overlap
            best_sense = sense.definition()
        for ex in sense.examples():
            ex = ex.split(' ')
            ex = remove_sw(ex)
            ex_we = get_sentence_word_embedding(ex)
            overlap = cosinus_distance(sentence_we, ex_we)
            if overlap > max_overlap:
                max_overlap = overlap
                best_sense = sense.definition()

    return best_sense

In [102]:
print('Best sense: ',WSD_with_embeddings('bank', 'Yesterday I went to the bank to withdraw the money and the credit card did not work'))

new best sense:  sloping land (especially the slope beside a body of water)
overlap:  0.22813908650223436

new best sense:  a financial institution that accepts deposits and channels the money into lending activities
overlap:  0.5661204138504646

new best sense:  put into a bank account
overlap:  0.6258607884937029

Best sense:  put into a bank account
